# dataloader-pin-memory-workers — ex1: build a DataLoader with num_workers and pin_memory configured

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `dataloader-pin-memory-workers`. Running the final beacon cell reports progress against the `PyTorch: DataLoader pin_memory + workers` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: DataLoader pin_memory + workers` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`dataloader-pin-memory-workers`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "dataloader-pin-memory-workers"
DD_SUBTOPIC = "PyTorch: DataLoader pin_memory + workers"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## PyTorch: `DataLoader` `pin_memory` + workers — quick refresher

Two `DataLoader` flags control input-pipeline throughput:

```python
loader = DataLoader(
    dataset,
    batch_size=64,
    num_workers=4,        # background processes that fetch batches
    pin_memory=True,      # batches land in page-locked RAM
    shuffle=True,
)
```

**`num_workers > 0` parallelizes data loading.** The main process yields a batch from a queue while N worker subprocesses prepare future batches in parallel. Bottleneck shifts from disk → CPU decode → main-thread-blocking to fully overlapped. `num_workers=4` is the common default; tune to ~half your CPU core count.

**`pin_memory=True` enables async CPU→GPU transfer.** Pinned (page-locked) RAM lets the GPU's DMA engine copy a batch while the previous batch is still being trained on. Combined with `batch.to(device, non_blocking=True)` in the training loop, you get essentially free CPU→GPU staging.

**Pin-memory caveat.** Page-locked RAM is a scarce resource; if you pin too much (huge batches × many workers) you can OOM the system RAM. The default `False` is conservative — turn it ON for GPU training, leave it OFF for CPU training (no benefit).

**Workers > 0 caveat.** Each worker is a separate Python process. Anything you pass through the dataset must be picklable. Lambdas in transforms break — use a top-level function. On Windows, `spawn` is the only start method, which is much slower to bring up.

### Exercise 1 — build a DataLoader with num_workers and pin_memory configured

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply the `DataLoader(dataset, batch_size, num_workers, pin_memory, shuffle)` constructor to build a configured loader and verify each argument propagated correctly.
> Keywords: dataloader, pin-memory, num-workers, throughput
> ```

**KCs targeted:** `dataloader-config-args`, `shuffle-vs-sampler-mutex`

Implement `ex1_make_dataloader(dataset, batch_size, num_workers, pin_memory, shuffle)`. A thin factory.

1. Construct and return a `torch.utils.data.DataLoader` with all five arguments wired through.
2. No defaults — the caller provides every value explicitly.

The test verifies:
- The DataLoader's attributes match the inputs.
- The DataLoader yields the correct number of batches.
- A passed shuffle=True actually re-orders batches across two iterations (deterministic shuffle is fine; we just check at least one permutation differs from the trivial identity).
- Setting `num_workers=0` runs single-process (the Colab-safe default for notebooks where pickling cell-defined classes fails).

Inputs:
- `dataset`: a `torch.utils.data.Dataset` instance.
- `batch_size`: int.
- `num_workers`: int >= 0.
- `pin_memory`: bool.
- `shuffle`: bool.

Output: `DataLoader`.

In [ ]:
from torch.utils.data import DataLoader

def ex1_make_dataloader(dataset, batch_size, num_workers, pin_memory, shuffle):
    return DataLoader(
        dataset,
        batch_size=batch_size,
        num_workers=num_workers,
        pin_memory=pin_memory,
        shuffle=shuffle,
    )


<details><summary>Solution</summary>

```python
from torch.utils.data import DataLoader

def ex1_make_dataloader(dataset, batch_size, num_workers, pin_memory, shuffle):
    return DataLoader(
        dataset,
        batch_size=batch_size,
        num_workers=num_workers,
        pin_memory=pin_memory,
        shuffle=shuffle,
    )
```

**`shuffle=True` and `sampler=` are mutually exclusive.** The DataLoader raises if you pass both. Internally `shuffle=True` constructs a `RandomSampler` for you; `shuffle=False` constructs a `SequentialSampler`. For distributed training you ALWAYS pass `sampler=` and OMIT `shuffle=` — the `DistributedSampler` handles the shuffle across ranks (see the `distributed-sampler-shard` drill).

**`pin_memory=False` is the right default for CPU.** Pinned memory only helps when there's a GPU transfer to overlap with. On CPU-only training it costs RAM and gains nothing.

**`num_workers=0` runs in the main process.** That's the Colab-safe default — multi-process loaders can fail to pickle objects defined in a notebook cell. `>0` is great for a real training script with top-level dataset classes.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()